In [1]:
# imports and class instantiations

import pandas as pd
import time
import xlwings as xw

from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo

from Options_Math_Helpers import *
from Options_Math_Algebra import *
from Options_Math_Black_Scholes import *
from Options_Skew_Wow_Alpha import *
from Options_Skew_SABR import *
from Options_Skew_SVI import *

omh  = OptionsMathHelpers()
oma  = OptionsMathAlgebra()
ombs = OptionsMathBlackScholes()
wa   = WowAlphaSkew()
sabr = SABRSkew()
svi  = SVISkew()

In [2]:
# get name of saved spreadsheet

def get_spreadsheet_name():    
    print("\n")    
    core_file_name = 'Daily_Deribit_Option_Snapshot_BTC_'
    answer = input("Date of Excel Workbook? (YYYYMMDD) ")
    workbook_name = core_file_name + answer + ".xlsx"
    print("\n")  
    return workbook_name 

workbook_name = get_spreadsheet_name()
wb = xw.Book(workbook_name)

Date of Excel Workbook? (YYYYMMDD)  20260127


In [3]:
# get data from existing saved spreadsheet

def get_input_dfs(key_names):    
    dict_ = {}
    for name in key_names:
        sheet_name = name + '_raw'
        worksheet = wb.sheets[sheet_name]
        dict_[name] = worksheet.used_range.options(pd.DataFrame, header=True, index=False).value
    return dict_

key_names   = ['futures', 'options']
dict_of_dfs = get_input_dfs(key_names)    

In [4]:
# add time, date and timedelta info to data

def create_dates_and_times(df):
    
    df['expiration_time_utc'], = omh.to_arrays(df['expiration_timestamp'], dtype='datetime64[ms]')  
    df['current_time_utc'], = omh.to_arrays(df['timestamp'], dtype='datetime64[ms]')  
    
    mask = df['expiration_time_utc'] > np.datetime64('2999-12-31')
    df.loc[mask, 'expiration_time_utc'] = np.datetime64('NaT')

    df['expiration_time_utc']  = df['expiration_time_utc'].dt.tz_localize("UTC")
    df['expiration_time_nyc']  = df['expiration_time_utc'].dt.tz_convert('America/New_York')

    df['current_time_utc']  = df['current_time_utc'].dt.tz_localize("UTC")
    df['current_time_nyc']  = df['current_time_utc'].dt.tz_convert('America/New_York')

    df['years_to_expiry']      = (df['expiration_time_nyc'] - df['current_time_nyc'].mean()) / pd.Timedelta(days=1)
    df['years_to_expiry']      = df['years_to_expiry'] / 365  #.25

    df['settlement_time_nyc']  = df['expiration_time_nyc']
    
    df['days_to_settlement']   = df['settlement_time_nyc'].dt.date - df['current_time_nyc'].dt.date
    df['days_to_settlement']   = pd.to_timedelta(df['days_to_settlement']) / pd.Timedelta(days=1)

    df['years_to_settlement']  = df['days_to_settlement'] / 360   # money market

    return df

for key, df in dict_of_dfs.items(): 
    dict_of_dfs[key] = create_dates_and_times(df.copy(deep=True))

In [5]:
# clean up and print data

def clean_up_data(df, wb, label):

    df = df.loc[:, df.columns.notna()] 
    df = df.dropna(how="all")
    
    df['years_to_expiry'],  = omh.to_arrays(df['years_to_expiry']) 

    if label == 'futures':
        df = df.sort_values(by=['expiration_timestamp']) 
        
    elif label == 'options':
        df['underlying_price'], = omh.to_arrays(df['underlying_price'])
        df['strike'], = omh.to_arrays(df['strike'])
        df['option_type'] = omh.option_type(df['option_type'], df['strike'])
        df = df.sort_values(by=['expiration_timestamp', 'strike', 'option_type'])
   
    return df

for key, df in dict_of_dfs.items(): 
    dict_of_dfs[key] = clean_up_data(df.copy(deep=True), wb, key)

In [6]:
# print routine used frequently throughout script

def print_df(df, wb, sheet_name, cols=None):
    if cols is not None:
        df = df.reindex(columns=cols)
    sheet = wb.sheets(sheet_name)
    sheet.range('a1').options(index = False, header=1).value = df 
    return df
    
for key, df in dict_of_dfs.items():
    sheet_name = key + "_with_time"
    return_df = print_df(df.copy(deep=True), wb, sheet_name)  
    if key == 'futures':
        futures_df = return_df
    else:
        options_df = return_df

In [7]:
# compare the deribit calculated greeks and vols to my own

def compare_greeks(df):
    
    df['mark_dollars'] = df['mark_price'] * df['underlying_price'] #'estimated_delivery_price'] #'underlying_price']

    opt_types        = df['option_type']
    fwds             = df['underlying_price'] #'estimated_delivery_price'] #'underlying_price']
    strikes          = df['strike']
    times            = df['years_to_expiry'] 
    
    df['my_mark_d0'] = ombs.bs_d0(fwd_value=fwds, strike=strikes, vol=df['mark_iv']/100, time_to_expiry=times)
    df['my_mark_d1'] = ombs.bs_d1(vol=df['mark_iv']/100, time_to_expiry=times, d0=df['my_mark_d0'])
    df['my_mark_d2'] = ombs.bs_d2(vol=df['mark_iv']/100, time_to_expiry=times, d0=df['my_mark_d0'])

    df['my_delta']   = ombs.bs_delta(opt_type=opt_types, fwd_value=fwds, strike=strikes, 
                                     vol=df['mark_iv']/100, time_to_expiry=times, d1=df['my_mark_d1'])
    df['my_gamma']   = ombs.bs_gamma(opt_type=opt_types, fwd_value=fwds, strike=strikes, 
                                     vol=df['mark_iv']/100, time_to_expiry=times, d1=df['my_mark_d1'])
    df['my_vega']    = ombs.bs_vega (opt_type=opt_types, fwd_value=fwds, strike=strikes, 
                                     vol=df['mark_iv']/100, time_to_expiry=times, d1=df['my_mark_d1'])
    df['my_theta']   = ombs.bs_theta(opt_type=opt_types, fwd_value=fwds, strike=strikes, 
                                     vol=df['mark_iv']/100, time_to_expiry=times, d1=df['my_mark_d1'], d2=df['my_mark_d2'])
    df['my_rho']     = 0 # ombs.bs_rho  (opt_type=opt_types, fwd=fwds, strike=strikes, 
#                                     vol=df['mark_iv']/100, time=times, d2=df['d2'])
#gotta think through theta and rho 
    df['my_iv']      = ombs.bs_vol_solver_newton(opt_value=df['mark_dollars'], opt_type=opt_types, 
                                                 fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100
    
    df['delta_diff'] = df['greeks_delta'] - df['my_delta']
    df['gamma_diff'] = df['greeks_gamma'] - df['my_gamma']
    df['vega_diff']  = df['greeks_vega']  - df['my_vega']
    df['theta_diff'] = df['greeks_theta'] - df['my_theta']
    df['rho_diff']   = df['greeks_rho']   - df['my_rho']
    df['iv_diff']    = df['mark_iv']      - df['my_iv']

    return df
    
comparison_df = options_df.copy(deep=True)
comparison_df = compare_greeks(comparison_df)

In [8]:
# print comparison df

cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',
        
        'mark_price', 
        'mark_dollars',

        'my_mark_d0',
        'my_mark_d1', 
        'my_mark_d2',           
        
        'mark_iv', 
        'my_iv', 
        'iv_diff', 
        
        'greeks_delta',
        'my_delta', 
        'delta_diff', 
        
        'greeks_gamma',
        'my_gamma', 
        'gamma_diff', 
        
        'greeks_vega',
        'my_vega', 
        'vega_diff', 
        
        'greeks_theta',
        'my_theta', 
        'theta_diff', 
        
        'greeks_rho',
        'my_rho', 
        'rho_diff'] 

sheet_name = 'option_calculation_comparisons'
comparison_df = print_df(comparison_df, wb, sheet_name, cols=cols)

In [9]:
# enrich option prices using time value and option arbitrage rules

def enrich_prices(df):
    
    opt_types        = df['option_type']
    fwds             = df['underlying_price'] #'estimated_delivery_price'] #'underlying_price']
    strikes          = df['strike']
    times            = df['years_to_expiry']    
    
    df['mark_dollars']     = df['underlying_price'] * df['mark_price']
    df['best_bid_dollars'] = df['underlying_price'] * df['best_bid_price']  
    df['best_ask_dollars'] = df['underlying_price'] * df['best_ask_price'] 

    df['best_bid_iv'] = ombs.bs_vol_solver_newton(opt_value=df['best_bid_dollars'], opt_type=opt_types, 
                                                  fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100   
    df['best_ask_iv'] = ombs.bs_vol_solver_newton(opt_value=df['best_ask_dollars'], opt_type=opt_types, 
                                                  fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100
    df['mark_iv']     = ombs.bs_vol_solver_newton(opt_value=df['mark_dollars'], opt_type=opt_types, 
                                                  fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100
    df['best_iv_spread'] = df['best_ask_iv'] - df['best_bid_iv']

    df['intrinsic'] = oma.intrinsic_value(opt_type=df['option_type'], 
                                          spot=df['underlying_price'], 
                                          strike_pv=df['strike']) # both are fv
    
    df['time_val_bid'] = df['best_bid_dollars'] - df['intrinsic']
    df['time_val_ask'] = df['best_ask_dollars'] - df['intrinsic']

    grid_df = omh.to_option_grid_df(df)

    grid_df["best_tv_strike_bid"] = grid_df[["time_val_bid_c", "time_val_bid_p"]].max(axis=1)
    grid_df["best_tv_strike_ask"] = grid_df[["time_val_ask_c", "time_val_ask_p"]].min(axis=1)
    
    cols_to_match = ["strike", "expiration_time_nyc"]
    
    df = df.merge(
                    grid_df[cols_to_match + ["best_tv_strike_bid", "best_tv_strike_ask"]],
                    on=cols_to_match,
                    how="left"
                 )
    
    df['best_tv_strike_bid'] = np.maximum(0, df['best_tv_strike_bid'])

    unique_expirations = df['expiration_time_nyc'].unique()

    hi_strike_mask = df['strike'] > df['underlying_price']
    lo_strike_mask = ~hi_strike_mask

    for exp in unique_expirations:
        exp_mask = df['expiration_time_nyc'] == exp
    
        # bids first
        joint_mask = lo_strike_mask & exp_mask
        tv_col = df.loc[joint_mask, 'best_tv_strike_bid']
        df.loc[joint_mask, 'enriched_bid_tv'] = tv_col.cummax()
    
        joint_mask = hi_strike_mask & exp_mask
        tv_col = df.loc[joint_mask, 'best_tv_strike_bid']
        df.loc[joint_mask, 'enriched_bid_tv'] = tv_col.iloc[::-1].cummax().iloc[::-1]
        
        # asks second
        joint_mask = hi_strike_mask & exp_mask
        tv_col = df.loc[joint_mask, 'best_tv_strike_ask']
        df.loc[joint_mask, 'enriched_ask_tv'] = tv_col.cummin()
        
        joint_mask = lo_strike_mask & exp_mask
        tv_col = df.loc[joint_mask, 'best_tv_strike_ask']
        df.loc[joint_mask, 'enriched_ask_tv'] = tv_col.iloc[::-1].cummin().iloc[::-1]   

    df['enriched_bid_price'] = df['intrinsic'] + df['enriched_bid_tv']
    df['enriched_ask_price'] = df['intrinsic'] + df['enriched_ask_tv']
    df['enriched_mid_price'] = (df['enriched_bid_price'] + df['enriched_ask_price']) / 2
    
    df['bid_improvement'] = df['enriched_bid_price'] - df['best_bid_dollars']   
    df['ask_improvement'] = df['best_ask_dollars'] - df['enriched_ask_price']

    opt_types        = df['option_type']
    fwds             = df['underlying_price'] #'estimated_delivery_price'] #'underlying_price']
    strikes          = df['strike']
    times            = df['years_to_expiry']    
   
    df['enriched_bid_iv'] = ombs.bs_vol_solver_newton(opt_value=df['enriched_bid_price'], opt_type=opt_types, 
                                                      fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100   
    df['enriched_ask_iv'] = ombs.bs_vol_solver_newton(opt_value=df['enriched_ask_price'], opt_type=opt_types, 
                                                      fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100
    df['enriched_mid_iv'] = ombs.bs_vol_solver_newton(opt_value=df['enriched_mid_price'], opt_type=opt_types, 
                                                      fwd_value=fwds, strike=strikes, time_to_expiry=times) * 100
    df['enriched_iv_spread'] = df['enriched_ask_iv'] - df['enriched_bid_iv']

    vols = df['enriched_mid_iv'] / 100
    
    d0 = ombs.bs_d0(fwd_value=fwds, strike=strikes, vol=vols, time_to_expiry=times)    
    d1 = ombs.bs_d1(vol=vols, time_to_expiry=times, d0=d0)        
    df['enriched_mid_vega'] = ombs.bs_vega (opt_type=opt_types, 
                                            fwd_value=fwds, 
                                            strike=strikes, 
                                            vol=vols, 
                                            time_to_expiry=times, 
                                            d1=d1)     

    return df

enrich_df = options_df.copy(deep=True)
enrich_df = enrich_prices(enrich_df) 

In [10]:
# print enriched df

cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',

        'best_bid_price', 
        'best_ask_price',
        'mark_price', 

        'best_bid_dollars',
        'best_ask_dollars',            
        'mark_dollars',

        'best_bid_iv', 
        'best_ask_iv', 
        'mark_iv', 
        'best_iv_spread',

        'intrinsic',
        'time_val_bid',
        'time_val_ask',

        'best_tv_strike_bid', 
        'best_tv_strike_ask',

        'enriched_bid_tv', 
        'enriched_ask_tv',            

        'enriched_bid_price',
        'enriched_ask_price',
        'enriched_mid_price',

        'bid_improvement',  
        'ask_improvement',

        'enriched_bid_iv',
        'enriched_ask_iv',
        'enriched_mid_iv',
        'enriched_iv_spread',
       
        'enriched_mid_vega',   
       ]

sheet_name = 'enriched_prices'
enrich_df = print_df(enrich_df, wb, sheet_name, cols=cols)

In [11]:
# calc combo prices from options prices

def calc_combo_prices(combos_df):

    combos_df = omh.to_option_grid_df(combos_df)

    combos_df['go_short_upfront_amt'] =  combos_df['best_bid_dollars_c'] -  combos_df['best_ask_dollars_p'] 
    combos_df['go_short_arrears_amt'] =  combos_df['strike']
    combos_df['go_short_net_amt']     =  combos_df['go_short_upfront_amt'] + combos_df['go_short_arrears_amt']
    
    combos_df['go_long_upfront_amt']  =  combos_df['best_bid_dollars_p'] -  combos_df['best_ask_dollars_c'] 
    combos_df['go_long_arrears_amt']  = -combos_df['strike']
    combos_df['go_long_net_amt']      =  combos_df['go_long_upfront_amt'] + combos_df['go_long_arrears_amt']

    return combos_df

combos_df = enrich_df.copy(deep=True)
combos_df = calc_combo_prices(combos_df)

In [12]:
# print combos df

cols = ['best_bid_dollars_c',
        'best_ask_dollars_c',	 
        'strike',
        'expiration_time_nyc',	 
        'best_bid_dollars_p', 	 
        'best_ask_dollars_p', 	 
        'go_short_upfront_amt', 	 
        'go_short_arrears_amt', 	 
        'go_short_net_amt', 	 
        'go_long_upfront_amt', 	 
        'go_long_arrears_amt', 	 
        'go_long_net_amt'] 

sheet_name = 'combo_prices'
combos_df = print_df(combos_df, wb, sheet_name, cols=cols)

In [13]:
# calc best fwd prices from futures and combos

def calc_best_fwds(combos_df, futures_df):
    
    df = pd.DataFrame({'expiration_time_nyc': combos_df['expiration_time_nyc'].unique()})

    for side, col in [('short', 'go_short_net_amt'), ('long', 'go_long_net_amt')]:
        best_idx = combos_df.groupby('expiration_time_nyc')[col].idxmax()
        best_rows = combos_df.loc[best_idx].set_index('expiration_time_nyc')
        df = df.merge(best_rows[['strike', col]].rename(columns={'strike': f'go_{side}_strike'}), 
                      left_on='expiration_time_nyc', right_index=True)

    df = df.merge(futures_df, on='expiration_time_nyc', how='left')
    df['best_ask_price'] = -df['best_ask_price']
    
    df['best_go_short_amt'] = df[['go_short_net_amt', 'best_bid_price']].max(axis=1)
    df['best_go_long_amt']  = df[['go_long_net_amt', 'best_ask_price']].max(axis=1)


    df['best_go_short_kind']   = np.where(
                                          df['best_go_short_amt'] == df['go_short_net_amt'], 
                                          'combo',  # value if condition is True
                                          df['kind']  # value if condition is False
                                         )
    df['best_go_long_kind']    = np.where(
                                          df['best_go_long_amt'] == df['go_long_net_amt'], 
                                          'combo',  # value if condition is True
                                          df['kind']  # value if condition is False
                                         )
    df['best_go_short_strike'] = np.where(
                                          df['best_go_short_amt'] == df['go_short_net_amt'], 
                                          df['go_short_strike'],  # value if condition is True
                                          None # value if condition is False
                                         )
    df['best_go_long_strike']  = np.where(
                                          df['best_go_long_amt'] == df['go_long_net_amt'], 
                                          df['go_long_strike'],  # value if condition is True
                                          None # value if condition is False
                                         )
    df['best_round_trip'] = df['best_go_short_amt'].astype('float') + df['best_go_long_amt'].astype('float')

    return df
           
best_fwds_df = calc_best_fwds(combos_df, futures_df)

In [14]:
# print best fwds df

cols = ['best_go_short_amt', 
        'best_go_short_kind',
        'best_go_short_strike',
        'expiration_time_nyc',
        'best_go_long_amt',
        'best_go_long_kind',
        'best_go_long_strike',
        'best_round_trip'
       ] 

sheet_name = 'best_fwd_prices'
best_fwds_df = print_df(best_fwds_df, wb, sheet_name, cols=cols)

In [15]:
# use enriched option prices to calculate wow alpha skew parameters

def calibrate_wow_alpha(df, initial_wow=0.4, initial_conv=-0.1):

    exp_list  = df['years_to_expiry'].unique()

    params_dicts_list = []    
    for exp in exp_list:

        exp_mask   = df['years_to_expiry'] == exp 
        
        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
           
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']
        
        vols      = df.loc[exp_mask, 'enriched_mid_iv'] / 100
        fwd_val   = fwds.mean()
        atm_vol   = np.interp(fwd_val, strikes, vols)
        
        tgt_vals  = df.loc[exp_mask, 'enriched_mid_price']
        
        vegas     = df.loc[exp_mask, 'enriched_mid_vega']
        
        wow, convexity, result = wa.calibrate_wow_alpha_weighted(opt_type=opt_types, 
                                                                 fwd=fwds, 
                                                                 strike=strikes, 
                                                                 time=times, 
                                                                 vol_atm=atm_vol,
                                                                 target_values=tgt_vals, 
                                                                 weighting='vega',
                                                                 weights=vegas,
                                                                 initial_guess=(initial_wow, initial_conv),
                                                                 bounds=([-10, -10], [10, 10]))
     
        params_dict = {'expiration_time_nyc' : exp_date, 'years_to_expiry' : exp, 
                       "atm_vol" : atm_vol, "wow" : wow, 'conv' : convexity}
        params_dicts_list.append(params_dict)        
    
    params_df = pd.DataFrame(params_dicts_list)
    
    for exp in exp_list:   
        exp_mask   = df['years_to_expiry'] == exp 

        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']

        atm_vol   = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'atm_vol'].to_list()[0]
        wow       = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'wow'].to_list()[0]
        convexity = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'conv'].to_list()[0]
        
        df.loc[exp_mask, 'wow_alpha_price'] = wa.wow_alpha_value(opt_type=opt_types, 
                                                                 fwd=fwds, 
                                                                 strike=strikes, 
                                                                 time=times, 
                                                                 vol_atm=atm_vol, 
                                                                 wow=wow, 
                                                                 convexity=convexity)

    df['wow_alpha_-_best_bid'] = df['wow_alpha_price'] - df['best_bid_dollars']
    df['wow_alpha_-_enriched_bid'] = df['wow_alpha_price'] - df['enriched_bid_price']
    df['enriched_ask_-_wow_alpha'] = df['enriched_ask_price'] - df['wow_alpha_price']   
    df['best_ask_-_wow_alpha'] = df['best_ask_dollars'] - df['wow_alpha_price']   

    return df, params_df

wow_alpha_df = enrich_df.copy(deep=True)
wow_alpha_df, skew_params_df = calibrate_wow_alpha(wow_alpha_df)

In [16]:
# print wow alpha df
   
cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',

        'best_bid_dollars',
        'best_ask_dollars',            
        'mark_dollars',

        'enriched_bid_price',
        'enriched_ask_price',
        'enriched_mid_price',

        'wow_alpha_price',

        'wow_alpha_-_best_bid',
        'wow_alpha_-_enriched_bid',
        'enriched_ask_-_wow_alpha',
        'best_ask_-_wow_alpha'             
       ]

sheet_name = 'wow_alpha_prices'
wow_alpha_df = print_df(wow_alpha_df, wb, sheet_name, cols=cols)

In [17]:
# use enriched option prices to calculate sabr skew parameters

def calibrate_sabr(df):
    
    exp_list  = df['years_to_expiry'].unique()  

    params_dicts_list = []    
    for exp in exp_list:

        exp_mask   = df['years_to_expiry'] == exp 
        
        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
            
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']        
        tgt_vols  = df.loc[exp_mask, 'enriched_mid_iv'] / 100
        vegas     = df.loc[exp_mask, 'enriched_mid_vega']
        
        alpha, beta, rho, nu, result = sabr.calibrate_sabr_weighted(fwd=fwds, 
                                                                    strike=strikes, 
                                                                    time=times, 
                                                                    target_vols=tgt_vols, 
                                                                    weighting='vega',
                                                                    weights=vegas,
                                                                    weight_eps=1e-8,
                                                                    beta=1.0)

        params_dict = {'expiration_time_nyc' : exp_date, 'years_to_expiry' : exp, 
                       "alpha" : alpha, "beta" : float(beta), 'rho' : rho, 'nu' : nu}
        params_dicts_list.append(params_dict)        
    
    params_df = pd.DataFrame(params_dicts_list)
    
    for exp in exp_list:   
        exp_mask   = df['years_to_expiry'] == exp 

        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']

        alphas    = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'alpha'].to_list()[0]
        betas     = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'beta'].to_list()[0]
        rhos      = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'rho'].to_list()[0]
        nus       = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'nu'].to_list()[0]
        
        df.loc[exp_mask, 'sabr_vol'] = sabr.sabr_vol(fwd=fwds, 
                                                     strike=strikes, 
                                                     time=times, 
                                                     alpha=alphas,
                                                     beta=betas,
                                                     rho=rhos,
                                                     nu=nus) * 100

    df['sabr_vol_-_best_bid_iv'] = df['sabr_vol'] - df['best_bid_iv']
    df['sabr_vol_-_enriched_bid_iv'] = df['sabr_vol'] - df['enriched_bid_iv']
    df['enriched_ask_iv_-_sabr_vol'] = df['enriched_ask_iv'] - df['sabr_vol']   
    df['best_ask_iv_-_sabr_vol'] = df['best_ask_iv'] - df['sabr_vol']   

    return df, params_df

sabr_df = enrich_df.copy(deep=True)
sabr_df, sabr_params_df = calibrate_sabr(sabr_df)
skew_params_df = skew_params_df.merge(sabr_params_df, on=['expiration_time_nyc', 'years_to_expiry'])

In [18]:
# print sabr skew df
  
cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',

        'best_bid_iv',
        'best_ask_iv',            
        'mark_iv',

        'enriched_bid_iv',
        'enriched_ask_iv',
        'enriched_mid_iv',

        'enriched_mid_vega',

        'sabr_vol',

        'sabr_vol_-_best_bid_iv',
        'sabr_vol_-_enriched_bid_iv',
        'enriched_ask_iv_-_sabr_vol',
        'best_ask_iv_-_sabr_vol'             
       ]

sheet_name = 'sabr_vols'
sabr_df = print_df(sabr_df, wb, sheet_name, cols=cols)

In [19]:
# use enriched option prices to calculate sabr skew parameters

def calibrate_svi(df):
    
    exp_list  = df['years_to_expiry'].unique()  

    params_dicts_list = []    
    for exp in exp_list:

        exp_mask   = df['years_to_expiry'] == exp 
        
        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
            
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']        
        tgt_vols  = df.loc[exp_mask, 'enriched_mid_iv'] / 100
        vegas     = df.loc[exp_mask, 'enriched_mid_vega']
        
        a, b, rho, m, sigma, result = svi.calibrate_svi_weighted(fwd=fwds, 
                                                                    strike=strikes, 
                                                                    time=times, 
                                                                    target_vols=tgt_vols, 
                                                                    weighting='vega',
                                                                    weights=vegas)

        params_dict = {'expiration_time_nyc' : exp_date, 'years_to_expiry' : exp, 
                       "a" : a, "b" : b, 'rho' : rho, 'm' : m, 'sigma' : sigma}
        params_dicts_list.append(params_dict)        
    
    params_df = pd.DataFrame(params_dicts_list)
    
    for exp in exp_list:   
        exp_mask   = df['years_to_expiry'] == exp 

        exp_date  = df.loc[exp_mask, 'expiration_time_nyc'].unique()[0]
        opt_types = df.loc[exp_mask, 'option_type']
        fwds      = df.loc[exp_mask, 'underlying_price']
        strikes   = df.loc[exp_mask, 'strike']
        times     = df.loc[exp_mask, 'years_to_expiry']

        a         = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'a'].to_list()[0]
        b         = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'b'].to_list()[0]
        rho       = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'rho'].to_list()[0]
        m         = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'm'].to_list()[0]
        sigma     = params_df.loc[params_df['expiration_time_nyc'] == exp_date, 'sigma'].to_list()[0]
        
        
        df.loc[exp_mask, 'svi_vol'] = svi.svi_vol(fwd=fwds, 
                                                  strike=strikes, 
                                                  time=times, 
                                                  a=a,
                                                  b=b,
                                                  rho=rho,
                                                  m = m,
                                                  sigma = sigma) * 100

    df['svi_vol_-_best_bid_iv'] = df['svi_vol'] - df['best_bid_iv']
    df['svi_vol_-_enriched_bid_iv'] = df['svi_vol'] - df['enriched_bid_iv']
    df['enriched_ask_iv_-_svi_vol'] = df['enriched_ask_iv'] - df['svi_vol']   
    df['best_ask_iv_-_svi_vol'] = df['best_ask_iv'] - df['svi_vol']   

    return df, params_df

svi_df = enrich_df.copy(deep=True)
svi_df, svi_params_df = calibrate_svi(svi_df)
skew_params_df = skew_params_df.merge(svi_params_df, on=['expiration_time_nyc', 'years_to_expiry'])

In [20]:
# print svi skew df
  
cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',

        'best_bid_iv',
        'best_ask_iv',            
        'mark_iv',

        'enriched_bid_iv',
        'enriched_ask_iv',
        'enriched_mid_iv',

        'enriched_mid_vega',

        'svi_vol',

        'svi_vol_-_best_bid_iv',
        'svi_vol_-_enriched_bid_iv',
        'enriched_ask_iv_-_svi_vol',
        'best_ask_iv_-_svi_vol'             
       ]

sheet_name = 'svi_vols'
svi_df = print_df(svi_df, wb, sheet_name, cols=cols)

In [21]:
# use atm vols to calculate forward vols

def calc_fwd_vols(df):

    new_row = pd.DataFrame([{col: '' for col in df.columns}])  
    new_row['years_to_expiry'] = 0.0
    new_row['atm_vol'] = 0.0
        
    df = pd.concat([df, new_row], ignore_index=True)
    df = df.sort_values('years_to_expiry')
    
    df['total_variance'] = (df['years_to_expiry'] * 
                            df['atm_vol'] * df['atm_vol'])
    df['fwd_variance'] = df['total_variance'].diff().shift(-1)
    
    df['fwd_time'] = df['years_to_expiry'].diff().shift(-1)
    
    df['fwd_vol']      = np.sqrt(df['fwd_variance'] / df['fwd_time'])

    return df

skew_params_df = calc_fwd_vols(skew_params_df)

In [22]:
# calibrate heston

heston_df = enrich_df.copy(deep=True)
#heston_df, heston_params_df = calibrate_heston(heston_df)
#skew_params_df = skew_params_df.merge(heston_params_df, on=['expiration_time_nyc', 'years_to_expiry'])

In [23]:
# print heston skew df

cols = ['instrument_name', 
        'option_type', 
        'strike', 
        
        'expiration_time_nyc', 
        'years_to_expiry', 
        
        'underlying_price',

        'best_bid_iv',
        'best_ask_iv',            
        'mark_iv',

        'enriched_bid_iv',
        'enriched_ask_iv',
        'enriched_mid_iv',

        'enriched_mid_vega',

#        'heston_vol',

 #       'heston_vol_-_best_bid_iv',
  #      'heston_vol_-_enriched_bid_iv',
   #     'enriched_ask_iv_-_heston_vol',
    #    'best_ask_iv_-_heston_vol'             
       ]

sheet_name = 'heston_vols'
heston_df = print_df(heston_df, wb, sheet_name, cols=cols)

In [24]:
### print skew params df

sheet_name = 'fwd_vols_and_skew_params'
skew_params_df = print_df(skew_params_df, wb, sheet_name)